# Proyecto de Curso: Vision Transformer para Detección de Rostros Reales vs. Falsos
## Etapas 1 y 2: Adquisición y Gestión de Datos + Preprocesamiento Paralelo

**Curso:** Programación Paralela y Distribuida  
**Profesor:** Johansell Villalobos Cubillo  
**Plataforma:** Ejecución local (PC personal)  
**Dataset:** [10000 Real vs Fake Faces (StyleGAN3)](https://www.kaggle.com/datasets/troykueh/real-vs-fake-faces-stylegan3)

---

### Justificación del uso de herramientas paralelas y distribuidas

El dataset contiene **10,000 imágenes RGB de alta resolución (1024×1024 px)**, distribuidas en dos clases (`Real` y `Fake`). El volumen total supera **1 GB** en disco. Las operaciones de carga, decodificación, redimensionamiento y normalización de imágenes son computacionalmente intensivas si se realizan de forma secuencial.

Para abordar este cuello de botella se emplean:
- **`Polars`** para el manejo eficiente y paralelo del inventario de archivos y metadatos (columnar, lazy evaluation, multi-hilo nativo).
- **`PyArrow`** para la serialización eficiente de metadatos en formato Parquet.
- **`Dask`** para operaciones distribuidas sobre el inventario de archivos.
- **`concurrent.futures.ProcessPoolExecutor`** para la decodificación y transformación paralela de imágenes usando todos los núcleos disponibles del CPU local (AMD Ryzen 5 5500, 6 núcleos / 12 hilos).
- **`torchvision` DataLoader** con workers paralelos integrados para el pipeline de entrenamiento.

## 0. Instalación de dependencias

In [ ]:
import subprocess, sys

packages = [
    "kaggle",
    "polars",
    "dask[dataframe]",
    "pyarrow",
    "Pillow",
    "tqdm",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "torch",
    "torchvision",
    "scikit-learn",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

print("✅ Todas las dependencias instaladas correctamente.")

## 1. Adquisición y Gestión de Datos

### 1.1 Configuración del API Token de Kaggle

La nueva API de Kaggle usa tokens `KGAT_*` guardados en `~/.kaggle/access_token` (no el `.json` clásico).  
Pega tu token en la variable `KAGGLE_TOKEN` o configúralo como variable de entorno antes de ejecutar.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

# ─── CONFIGURA TU TOKEN AQUÍ ──────────────────────────────────────────────────
KAGGLE_TOKEN = "KGAT_4f1c32ca18885ba4dd5a1a587bc40761"   # <-- pega tu token
# ─────────────────────────────────────────────────────────────────────────────

KAGGLE_DIR        = Path.home() / ".kaggle"
ACCESS_TOKEN_FILE = KAGGLE_DIR / "access_token"

KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
ACCESS_TOKEN_FILE.write_text(KAGGLE_TOKEN)
ACCESS_TOKEN_FILE.chmod(0o600)

# También exponerlo como variable de entorno (algunas versiones de kaggle lo leen así)
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

print(f"✅ Token configurado en: {ACCESS_TOKEN_FILE}")

# Verificar que el CLI de kaggle lo reconoce
result = subprocess.run(["kaggle", "config", "view"], capture_output=True, text=True)
print(result.stdout or result.stderr)

### 1.2 Descarga del dataset desde Kaggle

In [ ]:
# ── Rutas de trabajo ──────────────────────────────────────────────────────────
# Ejecución local: ajusta DATA_ROOT si deseas guardar en otra unidad/carpeta
DATA_ROOT = Path.home() / "proyecto_paralela" / "data"
RAW_DIR   = DATA_ROOT / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "troykueh/real-vs-fake-faces-stylegan3"

print(f"📁 Directorio de descarga : {RAW_DIR}")
print(f"🔗 Dataset                : {DATASET_SLUG}")
print("\n⏳ Iniciando descarga...")

t0 = time.time()
result = subprocess.run(
    ["kaggle", "datasets", "download", "-d", DATASET_SLUG,
     "--unzip", "-p", str(RAW_DIR)],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"✅ Descarga completada en {elapsed:.1f}s")
    print(result.stdout)
else:
    print("❌ Error durante la descarga:")
    print(result.stderr)

### 1.3 Detección de carpetas y verificación de la estructura

In [ ]:
import numpy as np

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# Detectar carpetas automáticamente (case-insensitive)
REAL_DIR, FAKE_DIR = None, None
for d in RAW_DIR.iterdir():
    if d.is_dir():
        if "real" in d.name.lower():
            REAL_DIR = d
        elif "fake" in d.name.lower():
            FAKE_DIR = d

assert REAL_DIR and FAKE_DIR, f"No se encontraron carpetas real/fake en {RAW_DIR}"

real_imgs = sorted([f for f in REAL_DIR.iterdir() if f.suffix.lower() in EXTENSIONS])
fake_imgs = sorted([f for f in FAKE_DIR.iterdir() if f.suffix.lower() in EXTENSIONS])

total_bytes = sum(f.stat().st_size for f in real_imgs + fake_imgs)

print(f"📂 Carpeta real : {REAL_DIR.name}")
print(f"📂 Carpeta fake : {FAKE_DIR.name}")
print(f"\n📊 Resumen del dataset:")
print(f"   Imágenes reales : {len(real_imgs):,}")
print(f"   Imágenes falsas : {len(fake_imgs):,}")
print(f"   Total           : {len(real_imgs) + len(fake_imgs):,}")
print(f"   Tamaño en disco : {total_bytes / 1e9:.2f} GB")

### 1.4 Construcción del inventario con Polars (paralelo nativo)

**Polars** ejecuta operaciones sobre DataFrames usando múltiples hilos de forma nativa (sin configuración adicional). Aquí lo usamos para construir y enriquecer el inventario de archivos de forma vectorizada.

In [ ]:
import polars as pl
import multiprocessing
import os

def get_allocated_cpus():
    """Detecta los CPUs realmente asignados por SLURM/cgroup, no los del nodo físico completo."""
    for var in ("SLURM_CPUS_PER_TASK", "SLURM_JOB_CPUS_PER_NODE", "SLURM_CPUS_ON_NODE"):
        val = os.environ.get(var)
        if val:
            try:
                return int(val.split("(")[0])
            except ValueError:
                pass
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:
        pass
    return multiprocessing.cpu_count()

N_WORKERS = get_allocated_cpus()
print(f"🖥️  Núcleos asignados (SLURM/cgroup) : {N_WORKERS}")
print(f"🧵  Threads Polars (automático)      : {pl.thread_pool_size()}")

# ── Construir registros base ───────────────────────────────────────────────────
t0 = time.time()

records = [
    {
        "filepath"  : str(f),
        "filename"  : f.name,
        "label"     : lbl,
        "class"     : cls,
        "size_bytes": f.stat().st_size,
        "ext"       : f.suffix.lower(),
    }
    for f, lbl, cls in
        [(p, 1, "real") for p in real_imgs] +
        [(p, 0, "fake") for p in fake_imgs]
]

# Crear DataFrame Polars y enriquecer con expresiones vectorizadas
df_meta = (
    pl.DataFrame(records)
    .with_columns([
        (pl.col("size_bytes") / 1024).round(2).alias("size_kb"),
        (pl.col("size_bytes") / 1e6).round(3).alias("size_mb"),
    ])
    .sort("filepath")
)

elapsed_build = time.time() - t0
print(f"\n✅ Inventario construido en {elapsed_build:.3f}s con Polars")
print(f"   Shape: {df_meta.shape}")
df_meta.head(5)

### 1.5 Análisis estadístico del inventario con Polars (Lazy API)

In [ ]:
# Lazy API de Polars: el plan de ejecución se optimiza antes de correr
stats_lazy = (
    df_meta.lazy()
    .group_by("class")
    .agg([
        pl.len().alias("n_images"),
        pl.col("size_kb").mean().round(2).alias("mean_kb"),
        pl.col("size_kb").std().round(2).alias("std_kb"),
        pl.col("size_kb").min().alias("min_kb"),
        pl.col("size_kb").max().alias("max_kb"),
        pl.col("size_kb").median().round(2).alias("median_kb"),
        pl.col("size_mb").sum().round(2).alias("total_mb"),
    ])
    .sort("class")
)

t0 = time.time()
df_stats = stats_lazy.collect()   # <- aquí se ejecuta el plan
print(f"⚡ Estadísticas calculadas en {time.time()-t0:.4f}s (Polars Lazy)\n")
print(df_stats)

### 1.6 Guardado del inventario en Parquet con PyArrow

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

META_DIR    = DATA_ROOT / "metadata"
PLOTS_DIR   = DATA_ROOT / "plots"
META_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = META_DIR / "dataset_inventory.parquet"

# Polars puede escribir Parquet directamente (usa PyArrow internamente)
df_meta.write_parquet(PARQUET_PATH, compression="snappy")

size_kb = PARQUET_PATH.stat().st_size / 1024
print(f"✅ Inventario guardado en : {PARQUET_PATH}")
print(f"   Tamaño Parquet         : {size_kb:.1f} KB (compresión Snappy)")

# Verificar lectura
df_check = pl.read_parquet(PARQUET_PATH)
print(f"   Verificación lectura   : {df_check.shape} ✅")

### 1.7 Análisis exploratorio visual

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convertir a pandas solo para las gráficas (seaborn lo requiere)
df_pd = df_meta.to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Análisis Exploratorio del Dataset (Metadatos)", fontsize=14, fontweight="bold")

# ── Distribución de clases ────────────────────────────────────────────────────
counts = df_pd["class"].value_counts()
axes[0].bar(counts.index, counts.values, color=["#2196F3", "#F44336"], edgecolor="black")
axes[0].set_title("Distribución de Clases")
axes[0].set_ylabel("Cantidad de imágenes")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha="center", fontweight="bold")

# ── Histograma de tamaños ──────────────────────────────────────────────────────
for label, color in [("real", "#2196F3"), ("fake", "#F44336")]:
    subset = df_pd[df_pd["class"] == label]["size_kb"]
    axes[1].hist(subset, bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title("Distribución del Tamaño (KB)")
axes[1].set_xlabel("Tamaño (KB)")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

# ── Boxplot por clase ─────────────────────────────────────────────────────────
sns.boxplot(data=df_pd, x="class", y="size_kb",
            palette={"real": "#2196F3", "fake": "#F44336"}, ax=axes[2])
axes[2].set_title("Tamaño por Clase (Boxplot)")
axes[2].set_ylabel("Tamaño (KB)")

plt.tight_layout()
fig.savefig(PLOTS_DIR / "eda_metadata.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura guardada.")

### 1.8 Visualización de muestras del dataset

In [ ]:
from PIL import Image
import random

random.seed(42)
sample_real = random.sample(real_imgs, 4)
sample_fake = random.sample(fake_imgs, 4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Muestras del Dataset: Real (arriba) vs Fake (abajo)",
             fontsize=14, fontweight="bold")

for i, (r_path, f_path) in enumerate(zip(sample_real, sample_fake)):
    img_real = Image.open(r_path).convert("RGB")
    img_fake = Image.open(f_path).convert("RGB")

    axes[0, i].imshow(img_real)
    axes[0, i].set_title(f"REAL ({img_real.size[0]}×{img_real.size[1]})",
                          color="#2196F3", fontweight="bold", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(img_fake)
    axes[1, i].set_title(f"FAKE ({img_fake.size[0]}×{img_fake.size[1]})",
                          color="#F44336", fontweight="bold", fontsize=9)
    axes[1, i].axis("off")

plt.tight_layout()
fig.savefig(PLOTS_DIR / "muestras_dataset.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figura guardada.")

---
## 2. Preprocesamiento Paralelo o Distribuido

Se implementa un pipeline de preprocesamiento con tres capas de paralelismo:

| Herramienta | Rol |
|---|---|
| **Polars** | Gestión y filtrado del inventario (multi-hilo nativo) |
| **ProcessPoolExecutor** | Decodificación + transformación de imágenes (multi-proceso) |
| **Dask** | Cómputo distribuido de estadísticas del canal RGB sobre los tensores |
| **PyTorch DataLoader** | Pipeline de mini-batches con `num_workers` paralelos |

### 2.1 Split estratificado con Polars

In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE    = 224
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Cargar inventario desde Parquet con Polars
df_inv = pl.read_parquet(PARQUET_PATH)

X = df_inv["filepath"].to_numpy()
y = df_inv["label"].to_numpy()

# Split estratificado 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=y, random_state=RANDOM_SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_SEED
)

# Guardar splits como Parquet con Polars
for split_name, paths, labels in [
    ("train", X_train, y_train),
    ("val",   X_val,   y_val),
    ("test",  X_test,  y_test),
]:
    pl.DataFrame({"filepath": paths, "label": labels}).write_parquet(
        META_DIR / f"split_{split_name}.parquet", compression="snappy"
    )

print(f"📊 Split del dataset (Polars + sklearn):")
print(f"   Train : {len(X_train):,}  (real: {y_train.sum():,} | fake: {(y_train==0).sum():,})")
print(f"   Val   : {len(X_val):,}   (real: {y_val.sum():,}  | fake: {(y_val==0).sum():,})")
print(f"   Test  : {len(X_test):,}   (real: {y_test.sum():,}  | fake: {(y_test==0).sum():,})")
print("\n✅ Splits guardados en Parquet.")

### 2.2 Función de preprocesamiento (worker)

**Nota especial para Windows:** a diferencia de Linux/Mac (que usan `fork`), Windows crea procesos nuevos con el método `spawn`. Esto significa que cada proceso hijo necesita poder **re-importar** la función worker desde un módulo real en disco — una función definida dentro de una celda de Jupyter no es importable por los procesos hijos y causa `BrokenProcessPool`.

Por eso la función `preprocess_image` vive en un archivo externo `preprocessing_worker.py` (en la misma carpeta que este notebook) y aquí simplemente la importamos.

In [ ]:
# La función preprocess_image vive en preprocessing_worker.py (mismo folder)
# Esto es OBLIGATORIO en Windows para que ProcessPoolExecutor funcione (método 'spawn').
import sys
from pathlib import Path

# Asegurar que el notebook encuentre el módulo aunque cambie el cwd
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from preprocessing_worker import preprocess_image

print("✅ Función worker importada desde preprocessing_worker.py")


### 2.3 Pipeline de preprocesamiento paralelo con ProcessPoolExecutor

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

PROCESSED_DIR = DATA_ROOT / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 500


def preprocess_split_parallel(paths, labels, split_name, is_train=False, n_workers=None):
    """Procesa un split completo en paralelo y guarda chunks .npy."""
    if n_workers is None:
        n_workers = multiprocessing.cpu_count()

    split_dir = PROCESSED_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    args_list    = list(zip(paths, labels, [is_train] * len(paths)))
    chunk_imgs   = []
    chunk_labels = []
    chunk_id     = 0
    errors       = 0

    print(f"\n🔄 Procesando split '{split_name}' | {len(paths):,} imgs | {n_workers} workers")
    t_start = time.time()

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(preprocess_image, a): a for a in args_list}
        pbar    = tqdm(total=len(args_list), desc=f"  {split_name:5s}", unit="img")

        for future in as_completed(futures):
            arr, lbl = future.result()
            pbar.update(1)

            if arr is None:
                errors += 1
                continue

            chunk_imgs.append(arr)
            chunk_labels.append(lbl)

            if len(chunk_imgs) >= CHUNK_SIZE:
                np.save(split_dir / f"images_chunk{chunk_id:03d}.npy", np.stack(chunk_imgs))
                np.save(split_dir / f"labels_chunk{chunk_id:03d}.npy", np.array(chunk_labels, dtype=np.int8))
                chunk_id   += 1
                chunk_imgs  = []
                chunk_labels= []

        pbar.close()

    # Último chunk
    if chunk_imgs:
        np.save(split_dir / f"images_chunk{chunk_id:03d}.npy", np.stack(chunk_imgs))
        np.save(split_dir / f"labels_chunk{chunk_id:03d}.npy", np.array(chunk_labels, dtype=np.int8))

    elapsed    = time.time() - t_start
    throughput = (len(paths) - errors) / elapsed

    print(f"   ✅ {elapsed:.1f}s | {throughput:.1f} img/s | Errores: {errors} | Chunks: {chunk_id+1}")
    return {"split": split_name, "total": len(paths)-errors, "errors": errors,
            "time_s": round(elapsed, 2), "throughput_img_s": round(throughput, 2),
            "n_chunks": chunk_id+1}


print(f"⚙️  Pipeline listo | {N_WORKERS} núcleos | chunk={CHUNK_SIZE} imgs")

In [ ]:
# Ejecutar preprocesamiento para los tres splits
# El guard __main__ es necesario en Windows: al usar 'spawn', el script
# se re-ejecuta en cada proceso hijo, y sin este guard se reiniciaría
# el pipeline completo de forma recursiva.
if __name__ == "__main__":
    bench_results = []
    bench_results.append(preprocess_split_parallel(X_train, y_train, "train", is_train=True,  n_workers=N_WORKERS))
    bench_results.append(preprocess_split_parallel(X_val,   y_val,   "val",   is_train=False, n_workers=N_WORKERS))
    bench_results.append(preprocess_split_parallel(X_test,  y_test,  "test",  is_train=False, n_workers=N_WORKERS))

    # Resultados en DataFrame Polars
    df_bench = pl.DataFrame(bench_results)
    print("\n📊 Resumen del preprocesamiento:")
    print(df_bench)


### 2.4 Estadísticas globales del dataset preprocesado con Dask

Se usan arrays de Dask para calcular media y desviación estándar por canal RGB sobre todo el split de entrenamiento de forma distribuida, verificando que la normalización fue correcta.

In [ ]:
import dask.array as da
from dask import delayed

train_dir = PROCESSED_DIR / "train"
img_files = sorted(train_dir.glob("images_chunk*.npy"))

print(f"📦 Cargando {len(img_files)} chunks con Dask Array...")

@delayed
def load_chunk(path):
    return np.load(path)

# Construir el array diferido (no carga en RAM todavía)
delayed_arrays = [da.from_delayed(
    load_chunk(f),
    shape=(CHUNK_SIZE, 3, IMG_SIZE, IMG_SIZE),
    dtype=np.float32
) for f in img_files]

# Concatenar a lo largo del eje 0 (imágenes)
X_dask = da.concatenate(delayed_arrays, axis=0)  # (N, C, H, W)

print(f"   Array Dask shape : {X_dask.shape}  (estimado)")
print(f"   Dtype            : {X_dask.dtype}")
print("\n⏳ Calculando estadísticas por canal (Dask distribuido)...")

t0 = time.time()
# Calcular mean/std por canal (eje 0,2,3 = N,H,W)
channel_mean = da.mean(X_dask, axis=(0, 2, 3)).compute()
channel_std  = da.std(X_dask,  axis=(0, 2, 3)).compute()
t_dask = time.time() - t0

print(f"✅ Estadísticas calculadas en {t_dask:.2f}s con Dask\n")
print(f"   Media por canal (R, G, B)   : {channel_mean.round(4)}")
print(f"   Std   por canal (R, G, B)   : {channel_std.round(4)}")
print(f"\n   Esperado (ImageNet mean)    : {IMAGENET_MEAN}")
print(f"   Esperado (ImageNet std)     : {IMAGENET_STD}")
print("\n   (Valores cercanos a 0 confirman que la normalización fue correcta)")

### 2.5 Benchmark: Serial vs Paralelo

In [ ]:
# Nota Windows: este bloque también usa ProcessPoolExecutor, por lo que
# necesita estar protegido con el guard __main__ (ver explicación en 2.3).
if __name__ == "__main__":
    BENCHMARK_SAMPLE = 200
    sample_paths  = X_train[:BENCHMARK_SAMPLE]
    sample_labels = y_train[:BENCHMARK_SAMPLE]
    sample_args   = list(zip(sample_paths, sample_labels, [False]*BENCHMARK_SAMPLE))

    print(f"⏱️  Benchmarking serial vs paralelo ({BENCHMARK_SAMPLE} imágenes)...")

    # Serial
    t0 = time.time()
    for a in sample_args:
        preprocess_image(a)
    t_serial = time.time() - t0

    # Paralelo
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        list(ex.map(preprocess_image, sample_args))
    t_parallel = time.time() - t0

    speedup    = t_serial / t_parallel
    efficiency = speedup / N_WORKERS * 100

    print(f"\n📊 Benchmark ({BENCHMARK_SAMPLE} imgs | {N_WORKERS} núcleos):")
    print(f"   Tiempo serial   : {t_serial:.2f}s  ({BENCHMARK_SAMPLE/t_serial:.1f} img/s)")
    print(f"   Tiempo paralelo : {t_parallel:.2f}s  ({BENCHMARK_SAMPLE/t_parallel:.1f} img/s)")
    print(f"   Speedup         : {speedup:.2f}×")
    print(f"   Eficiencia      : {efficiency:.1f}%")

    # Guardar métricas en Polars
    df_perf = pl.DataFrame({
        "method"      : ["serial", "paralelo"],
        "n_images"    : [BENCHMARK_SAMPLE, BENCHMARK_SAMPLE],
        "n_workers"   : [1, N_WORKERS],
        "time_s"      : [round(t_serial, 4), round(t_parallel, 4)],
        "throughput"  : [round(BENCHMARK_SAMPLE/t_serial, 2), round(BENCHMARK_SAMPLE/t_parallel, 2)],
        "speedup"     : [1.0, round(speedup, 4)],
        "efficiency_pct": [100.0, round(efficiency, 2)],
    })
    df_perf.write_parquet(META_DIR / "benchmark_serial_vs_paralelo.parquet")
    print("\n", df_perf)


### 2.6 Visualización del benchmark

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Benchmark: Preprocesamiento Serial vs Paralelo",
             fontsize=14, fontweight="bold")

methods = ["Serial\n(1 núcleo)", f"Paralelo\n({N_WORKERS} núcleos)"]
colors  = ["#EF5350", "#42A5F5"]

# ── Tiempo ────────────────────────────────────────────────────────────────────
bars = axes[0].bar(methods, [t_serial, t_parallel], color=colors, edgecolor="black", width=0.5)
for bar, v in zip(bars, [t_serial, t_parallel]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.05,
                 f"{v:.2f}s", ha="center", fontweight="bold")
axes[0].set_title("Tiempo de Ejecución")
axes[0].set_ylabel("Segundos")
axes[0].set_ylim(0, max(t_serial, t_parallel)*1.3)

# ── Throughput ────────────────────────────────────────────────────────────────
tp = [BENCHMARK_SAMPLE/t_serial, BENCHMARK_SAMPLE/t_parallel]
bars2 = axes[1].bar(methods, tp, color=colors, edgecolor="black", width=0.5)
for bar, v in zip(bars2, tp):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                 f"{v:.1f}\nimg/s", ha="center", fontweight="bold")
axes[1].set_title("Throughput")
axes[1].set_ylabel("Imágenes / segundo")
axes[1].set_ylim(0, max(tp)*1.3)

# ── Speedup & Eficiencia ──────────────────────────────────────────────────────
bar3 = axes[2].bar(["Speedup", "Eficiencia (%)"],
                    [speedup, efficiency],
                    color=["#66BB6A", "#FFA726"], edgecolor="black", width=0.5)
for bar, v in zip(bar3, [speedup, efficiency]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.3,
                 f"{v:.2f}", ha="center", fontweight="bold")
axes[2].set_title(f"Métricas de Paralelismo ({N_WORKERS} núcleos)")
axes[2].set_ylim(0, max(speedup, efficiency)*1.3)
axes[2].axhline(N_WORKERS, color="gray", linestyle="--", linewidth=1, label=f"Speedup ideal ({N_WORKERS}×)")
axes[2].legend(fontsize=8)

plt.tight_layout()
fig.savefig(PLOTS_DIR / "benchmark_serial_vs_paralelo.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Gráfica guardada.")

### 2.7 Dataset PyTorch y DataLoader

**Limitación importante en Windows + Jupyter:** PyTorch `DataLoader` con `num_workers > 0` usa multiprocessing con método `spawn`. En Windows, `spawn` necesita re-importar el script "principal" en cada proceso hijo. Cuando ese "principal" es un notebook de Jupyter (no un archivo `.py`), el proceso hijo intenta serializar el kernel de IPython completo y falla con `OSError: [Errno 22] Invalid argument` — el guard `if __name__ == "__main__":` no resuelve este caso particular, a diferencia de `ProcessPoolExecutor` en la sección 2.3.

**Por eso aquí, dentro del notebook, se usa `num_workers=0`** (carga en el proceso principal, sin paralelismo extra — el preprocesamiento pesado ya se hizo en la sección 2.3 con `ProcessPoolExecutor`, así que esta carga es liviana). Para producción/entrenamiento real con `num_workers > 0`, se incluye el script standalone `test_dataloader.py` que se ejecuta desde terminal (`python test_dataloader.py`), donde sí funciona el multiprocessing completo de PyTorch en Windows.

In [ ]:
# ChunkedNpyDataset vive en preprocessing_worker.py (mismo folder del notebook).
import torch
from torch.utils.data import DataLoader
from preprocessing_worker import ChunkedNpyDataset

BATCH_SIZE  = 32
NUM_WORKERS = 0   # 0 = sin multiprocessing extra; ver nota arriba sobre Windows+Jupyter

train_ds = ChunkedNpyDataset(PROCESSED_DIR / "train")
val_ds   = ChunkedNpyDataset(PROCESSED_DIR / "val")
test_ds  = ChunkedNpyDataset(PROCESSED_DIR / "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

imgs_b, lbls_b = next(iter(train_loader))

print(f"✅ Datasets y DataLoaders listos:")
print(f"   Train  : {len(train_ds):,} muestras | {len(train_loader)} batches")
print(f"   Val    : {len(val_ds):,}  muestras | {len(val_loader)} batches")
print(f"   Test   : {len(test_ds):,}  muestras | {len(test_loader)} batches")
print(f"\n   Batch shape  : {tuple(imgs_b.shape)}  (N, C, H, W)")
print(f"   Labels shape : {tuple(lbls_b.shape)}")
print(f"   Num workers  : {NUM_WORKERS} (dentro de Jupyter)")
print(f"\n   💡 Para entrenamiento con num_workers>0 real, usar: python test_dataloader.py")


### 2.8 Resumen y verificación final

In [ ]:
processed_bytes = sum(f.stat().st_size for f in PROCESSED_DIR.rglob("*.npy"))
raw_bytes       = sum(f.stat().st_size for f in real_imgs + fake_imgs)

print("═" * 60)
print("  RESUMEN — ETAPAS 1 y 2 COMPLETADAS")
print("═" * 60)
print(f"\n  Herramientas utilizadas:")
print(f"    • Polars {pl.__version__}      — inventario + stats (multi-hilo nativo)")
print(f"    • PyArrow               — serialización Parquet (Snappy)")
print(f"    • Dask Array            — estadísticas RGB distribuidas")
print(f"    • ProcessPoolExecutor   — decodificación paralela (multi-proceso)")
print(f"    • PyTorch DataLoader    — pipeline de batches con {NUM_WORKERS} workers")
print(f"\n  Dataset:")
print(f"    Datos crudos        : {raw_bytes/1e9:.2f} GB")
print(f"    Datos procesados    : {processed_bytes/1e9:.2f} GB  (.npy, float32)")
print(f"    Total imágenes      : {len(real_imgs)+len(fake_imgs):,}")
print(f"    Train/Val/Test      : {len(train_ds):,} / {len(val_ds):,} / {len(test_ds):,}")
print(f"\n  Rendimiento paralelo:")
print(f"    Núcleos CPU local   : {N_WORKERS}")
print(f"    Speedup medido      : {speedup:.2f}×")
print(f"    Eficiencia paralela : {efficiency:.1f}%")
print(f"    Throughput paralelo : {BENCHMARK_SAMPLE/t_parallel:.1f} img/s")
print(f"\n  Salida:")
print(f"    Tensores: (C=3, H=224, W=224) float32, normalizados ImageNet")
print(f"    Formato : .npy particionado (chunks de {CHUNK_SIZE} imgs)")
print(f"\n  ✅ Pipeline listo para Etapa 3: ViT + Entrenamiento")
print("═" * 60)